<a href="https://colab.research.google.com/github/imagra93/ML-course-labs/blob/main/machine_learning/lab%20-%20Decision%20Trees%20from%20scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Trees from Scratch

**Theory:** [notes_ML/06_decision_trees_cart.ipynb](../notes_ML/06_decision_trees_cart.ipynb) (impurity measures, split search, the CART algorithm, regression trees, pruning). This lab only implements it.

We build a small CART tree with NumPy only, then check it against `scikit-learn`. Steps:

1. Impurity functions (Gini, entropy, variance)
2. Search for the best split
3. Grow the tree recursively and predict by walking down it
4. Wrap it in a `fit` / `predict` class and compare with `scikit-learn` (classification and regression)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

# Only used for benchmarking against our implementation and for datasets —
# no scikit-learn tree code is used inside DecisionTreeScratch itself.
from sklearn.datasets import make_blobs, load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(42)

## 1. Impurity Measures

Impurity is one number that says how mixed the labels in a node are. With $p_k$ the fraction of class $k$ in the node:

$$I_{\text{Gini}} = 1 - \sum_k p_k^2 \qquad I_{\text{entropy}} = -\sum_k p_k \log_2 p_k \qquad I_{\text{MSE}} = \frac{1}{N}\sum_i (y_i - \bar y)^2$$

Gini and entropy are for classification, MSE (the variance) for regression. All three are $0$ for a pure node.

In code: 1) count each class, 2) turn counts into fractions $p_k$, 3) apply the formula.

In [ ]:
def gini(y):
    """Gini impurity of a label array."""
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1.0 - np.sum(p ** 2)


def entropy(y):
    """Shannon entropy (base 2) of a label array."""
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return max(0.0, -np.sum(p * np.log2(p)))


def mse_impurity(y):
    """Variance of a (regression) target array."""
    if len(y) == 0:
        return 0.0
    return np.mean((y - np.mean(y)) ** 2)


pure = np.array([0, 0, 0, 0])
mixed = np.array([0, 0, 1, 1])

print(f"Pure node   — Gini: {gini(pure):.3f}  Entropy: {entropy(pure):.3f}")
print(f"50/50 node  — Gini: {gini(mixed):.3f}  Entropy: {entropy(mixed):.3f}")

A pure node has zero impurity. A 50/50 binary node gets the maximum: $0.5$ for Gini and $1$ bit for entropy (see notes 06, section 3, for the full comparison).

## 2. Representing a Tree: the `Node` Class

Each **internal node** stores a split (feature `j`, threshold) and two children. Each **leaf** stores a prediction. Internal nodes also keep `n_samples` and `impurity_decrease`, which we reuse at the end to compute feature importances.

In [ ]:
class Node:
    """A node in a binary decision tree.

    Internal node: feature_index, threshold, left, right are set (value is None).
    Leaf node: value is set (everything else is None).
    """

    def __init__(self, feature_index=None, threshold=None, left=None, right=None, *,
                 value=None, n_samples=None, impurity_decrease=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.n_samples = n_samples
        self.impurity_decrease = impurity_decrease

    def is_leaf(self):
        return self.value is not None

## 3. Finding the Best Split

A split sends samples with $x_j \le \theta$ left and the rest right. Its score is the **impurity gain**: parent impurity minus the size-weighted impurity of the children (see notes 06, section 4):

$$G(j,\theta) = I(S) - \left(\frac{N_L}{N} I(S_L) + \frac{N_R}{N} I(S_R)\right)$$

In code: 1) for each feature, take midpoints between consecutive sorted unique values as candidate thresholds, 2) split the samples and compute the gain, 3) keep the pair with the largest gain.

In [ ]:
def best_split(X, y, impurity_fn):
    """Search all (feature, threshold) pairs for the one with the highest impurity gain.

    Returns (feature_index, threshold, gain); feature_index is None if no
    split improves on the parent's impurity.
    """
    m, n_features = X.shape
    parent_impurity = impurity_fn(y)

    best_gain = 0.0
    best_feature, best_threshold = None, None

    for feature in range(n_features):
        values = np.unique(X[:, feature])
        if len(values) < 2:
            continue
        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:
            left_mask = X[:, feature] <= threshold
            right_mask = ~left_mask
            n_left, n_right = left_mask.sum(), right_mask.sum()
            if n_left == 0 or n_right == 0:
                continue

            weighted_child_impurity = (
                n_left / m * impurity_fn(y[left_mask]) +
                n_right / m * impurity_fn(y[right_mask])
            )
            gain = parent_impurity - weighted_child_impurity

            if gain > best_gain:
                best_gain, best_feature, best_threshold = gain, feature, threshold

    return best_feature, best_threshold, best_gain

In [ ]:
# A toy dataset with an obvious clean split at x = 5
X_toy = np.array([[0.], [1.], [2.], [8.], [9.], [10.]])
y_toy = np.array([0, 0, 0, 1, 1, 1])

feature, threshold, gain = best_split(X_toy, y_toy, gini)
print(f"Best split — feature: {feature}, threshold: {threshold}, gain: {gain:.3f}")

The search finds $\theta = 5.0$ (the midpoint between $2$ and $8$). The parent Gini of $0.5$ drops to $0$ in both children, so the gain is $0.5$.

A less clean case (the worked example from notes 06, section 6): plotting the gain of every candidate threshold shows what `best_split` is maximising.

In [ ]:
# 8 points, one feature, one "noisy" label at x = 4 and x = 5
X_ex = np.arange(1, 9, dtype=float)[:, None]
y_ex = np.array([0, 0, 0, 1, 0, 1, 1, 1])

# Gain of every candidate threshold (midpoints between consecutive values)
thresholds = (X_ex[:-1, 0] + X_ex[1:, 0]) / 2
gains = []
for t in thresholds:
    left = X_ex[:, 0] <= t
    weighted = left.mean() * gini(y_ex[left]) + (~left).mean() * gini(y_ex[~left])
    gains.append(gini(y_ex) - weighted)

feature, threshold, gain = best_split(X_ex, y_ex, gini)
print(f"best_split -> threshold: {threshold}, gain: {gain:.3f}")

plt.figure(figsize=(7, 3.5))
plt.bar(thresholds, gains, width=0.6, color='steelblue')
plt.scatter(X_ex[:, 0], np.full(len(y_ex), -0.03), c=y_ex, cmap='coolwarm', s=80, edgecolors='k', zorder=3)
plt.axhline(0, color='grey', linewidth=0.8)
plt.xlabel(r'threshold $\theta$')
plt.ylabel('Gini gain')
plt.title('Gain of each candidate split (dots = samples, colour = class)')
plt.tight_layout()
plt.show()

Two thresholds tie at the top ($3.5$ and $5.5$, gain $0.3$). `best_split` keeps the first one it finds, because it only replaces the best split when the gain is strictly larger.

## 4. Growing the Tree Recursively

Starting from the root (all training samples):

1. Make a **leaf** if the node is pure, has fewer than `min_samples_split` samples, has reached `max_depth`, or no split gains at least `min_impurity_decrease`.
2. Otherwise, find the best split, partition the samples and recurse on each side.

A leaf predicts the **majority class** (classification) or the **mean target** (regression).

In [ ]:
def majority_class(y):
    """Most frequent label in y (ties broken towards the smaller label)."""
    values, counts = np.unique(y, return_counts=True)
    return values[np.argmax(counts)]


def build_tree(X, y, impurity_fn, leaf_value_fn, max_depth=None,
               min_samples_split=2, min_impurity_decrease=1e-7, depth=0):
    """Recursively grow a CART tree and return its root Node."""
    if (len(y) < min_samples_split
            or (max_depth is not None and depth >= max_depth)
            or impurity_fn(y) == 0.0):
        return Node(value=leaf_value_fn(y))

    feature, threshold, gain = best_split(X, y, impurity_fn)
    if feature is None or gain < min_impurity_decrease:
        return Node(value=leaf_value_fn(y))

    left_mask = X[:, feature] <= threshold
    right_mask = ~left_mask

    left = build_tree(X[left_mask], y[left_mask], impurity_fn, leaf_value_fn,
                       max_depth, min_samples_split, min_impurity_decrease, depth + 1)
    right = build_tree(X[right_mask], y[right_mask], impurity_fn, leaf_value_fn,
                        max_depth, min_samples_split, min_impurity_decrease, depth + 1)

    return Node(feature_index=feature, threshold=threshold, left=left, right=right,
                n_samples=len(y), impurity_decrease=gain)

## 5. Predicting: Traversing the Tree

To predict one sample, start at the root and go left if $x_j \le \theta$, right otherwise, until a leaf. `print_tree` shows the learned tree as nested `if / else` rules.

In [ ]:
def predict_sample(node, x):
    while not node.is_leaf():
        if x[node.feature_index] <= node.threshold:
            node = node.left
        else:
            node = node.right
    return node.value


def print_tree(node, feature_names=None, depth=0):
    indent = "  " * depth
    if node.is_leaf():
        value = f"{node.value:.3f}" if isinstance(node.value, float) else str(node.value)
        print(f"{indent}predict -> {value}")
        return

    name = feature_names[node.feature_index] if feature_names is not None else f"X[{node.feature_index}]"
    print(f"{indent}if {name} <= {node.threshold:.3f}:")
    print_tree(node.left, feature_names, depth + 1)
    print(f"{indent}else:")
    print_tree(node.right, feature_names, depth + 1)

## 6. Wrapping It Up: a Scikit-Learn-Style Class

The same builder handles both tasks. Only two things change:

| | Classification | Regression |
|---|---|---|
| Impurity | Gini or entropy | Variance (MSE) |
| Leaf value | Majority class | Mean target |

In [ ]:
class DecisionTreeScratch:
    """A minimal CART implementation for classification and regression.

    Parameters
    ----------
    task : {'classification', 'regression'}
    criterion : callable, optional
        Impurity function. Defaults to `gini` for classification and
        `mse_impurity` for regression.
    max_depth, min_samples_split, min_impurity_decrease : as in `build_tree`.
    """

    def __init__(self, task='classification', criterion=None, max_depth=None,
                 min_samples_split=2, min_impurity_decrease=1e-7):
        self.task = task
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_impurity_decrease = min_impurity_decrease

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        impurity_fn = self.criterion or (gini if self.task == 'classification' else mse_impurity)
        leaf_value_fn = majority_class if self.task == 'classification' else np.mean

        self.root_ = build_tree(X, y, impurity_fn, leaf_value_fn, self.max_depth,
                                 self.min_samples_split, self.min_impurity_decrease)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([predict_sample(self.root_, x) for x in X])

## 7. Sanity Check: a Toy 2D Dataset

Four 2D Gaussian blobs, so we can see the decision boundaries of both implementations side by side.

In [ ]:
def plot_decision_boundary(model, X, y, ax=None, title=None, cmap='rainbow'):
    ax = ax or plt.gca()
    pad = 1.0
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    n_classes = len(np.unique(y))

    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap, levels=np.arange(n_classes + 1) - 0.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, s=40, edgecolors='white', linewidths=0.5)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    if title:
        ax.set_title(title, fontsize=11)
    return ax

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, random_state=0, cluster_std=1.0)

scratch_tree = DecisionTreeScratch(task='classification', max_depth=4).fit(X_blobs, y_blobs)
sklearn_tree = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=0).fit(X_blobs, y_blobs)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(scratch_tree, X_blobs, y_blobs, ax=ax[0], title='DecisionTreeScratch (max_depth=4)')
plot_decision_boundary(sklearn_tree, X_blobs, y_blobs, ax=ax[1], title='scikit-learn DecisionTreeClassifier (max_depth=4)')
plt.tight_layout()
plt.show()

agreement = np.mean(scratch_tree.predict(X_blobs) == sklearn_tree.predict(X_blobs))
print(f"Predictions agree on {agreement:.1%} of training points")

Both boundaries are the same set of axis-aligned rectangles: the same greedy search with the same criterion. Any rare disagreement comes from ties (two equally good thresholds) broken differently.

Printing a depth-2 tree shows the rules it learned:

In [ ]:
small_tree = DecisionTreeScratch(task='classification', max_depth=2).fit(X_blobs, y_blobs)
print_tree(small_tree.root_)

## 8. Classification Benchmark: Iris

Test accuracy on Iris for several depths, scratch vs. `scikit-learn`.

In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)

print(f"{'max_depth':>10} | {'scratch acc':>12} | {'sklearn acc':>12}")
print("-" * 40)
for depth in [1, 2, 3, 5, None]:
    scratch = DecisionTreeScratch(task='classification', max_depth=depth).fit(X_train, y_train)
    sk = DecisionTreeClassifier(criterion='gini', max_depth=depth, random_state=42).fit(X_train, y_train)

    acc_scratch = accuracy_score(y_test, scratch.predict(X_test))
    acc_sk = accuracy_score(y_test, sk.predict(X_test))
    print(f"{str(depth):>10} | {acc_scratch:>12.3f} | {acc_sk:>12.3f}")

The two columns match at depths 1 and 2. From depth 3 on they differ by one or two test points (45 in total): when two splits have exactly the same gain, our code keeps the first one it finds, while `scikit-learn` visits features in a random order. The trees are equally good on the training data, but can generalise slightly differently.

## 9. Regression: Variance Reduction

For regression we use `mse_impurity` and `np.mean` as leaf value. The rest is identical. On a noisy 1D function a tree can only produce a **step function** (see notes 06, section 8).

In [ ]:
rng = np.random.RandomState(42)
x = 10 * rng.rand(200)

def target_fn(x, sigma=0.3):
    return np.sin(0.2 * np.pi * x) + np.sin(np.pi * x) + sigma * rng.randn(len(x))

y = target_fn(x)
X = x[:, None]

x_fit = np.linspace(0, 10, 1000)
y_true = target_fn(x_fit, sigma=0.0)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for a, depth in zip(ax, [3, 8]):
    scratch_reg = DecisionTreeScratch(task='regression', max_depth=depth).fit(X, y)
    sk_reg = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X, y)

    a.scatter(x, y, s=15, alpha=0.4, color='steelblue', label='Training data')
    a.plot(x_fit, y_true, color='grey', linewidth=1.5, linestyle='--', label='True function')
    a.plot(x_fit, scratch_reg.predict(x_fit[:, None]), color='crimson', linewidth=2, label='DecisionTreeScratch')
    a.plot(x_fit, sk_reg.predict(x_fit[:, None]), color='darkorange', linewidth=1.5, linestyle=':', label='sklearn')
    a.set_title(f'max_depth = {depth}', fontsize=11)
    a.legend(fontsize=8)

plt.tight_layout()
plt.show()

Both implementations give the same steps. `max_depth=3` allows at most $2^3 = 8$ flat segments and misses the fast oscillation (underfitting). `max_depth=8` starts to follow the noise (overfitting).

## 10. Overfitting and `max_depth`

Train vs. test accuracy on Iris as the tree gets deeper.

In [ ]:
depths = range(1, 11)
train_acc, test_acc = [], []

for depth in depths:
    model = DecisionTreeScratch(task='classification', max_depth=depth).fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, model.predict(X_train)))
    test_acc.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(7, 4.5))
plt.plot(list(depths), train_acc, marker='o', color='steelblue', label='Train accuracy')
plt.plot(list(depths), test_acc, marker='o', color='crimson', label='Test accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('DecisionTreeScratch — Overfitting vs. max_depth (Iris)')
plt.legend()
plt.tight_layout()
plt.show()

Train accuracy climbs to $1.0$ (a deep enough tree can make every leaf pure), while test accuracy peaks at depth 3-4 ($0.933$) and then drops slightly ($0.911$). That gap is overfitting, and `max_depth` is the simplest way to control it. Other stopping rules and cost-complexity pruning are covered in notes 06, section 9.

## Bonus: Feature Importance From Scratch

`scikit-learn`'s `feature_importances_` (Mean Decrease in Impurity): for each feature, add up `n_samples * impurity_decrease` over every node that splits on it, then normalise so the importances sum to $1$.

In code: 1) walk the tree, 2) add each internal node's contribution to its feature, 3) divide by the total.

In [ ]:
def feature_importances(root, n_features):
    importances = np.zeros(n_features)

    def walk(node):
        if node.is_leaf():
            return
        importances[node.feature_index] += node.n_samples * node.impurity_decrease
        walk(node.left)
        walk(node.right)

    walk(root)
    total = importances.sum()
    return importances / total if total > 0 else importances


iris_tree_scratch = DecisionTreeScratch(task='classification').fit(iris.data, iris.target)
iris_tree_sklearn = DecisionTreeClassifier(random_state=42).fit(iris.data, iris.target)

scratch_importances = feature_importances(iris_tree_scratch.root_, n_features=iris.data.shape[1])

print(f"{'Feature':<20} | {'Scratch':>8} | {'sklearn':>8}")
print("-" * 45)
for name, imp_s, imp_sk in zip(iris.feature_names, scratch_importances, iris_tree_sklearn.feature_importances_):
    print(f"{name:<20} | {imp_s:>8.3f} | {imp_sk:>8.3f}")

Both put almost all the importance on the petal measurements, which is where the Iris species separate. Small differences come from ties broken differently deep in the tree.

## Summary

- A tree is a set of axis-aligned splits with a constant prediction in each leaf.
- CART grows it greedily: at each node, pick the split with the largest impurity gain.
- Classification and regression use the same algorithm; only the impurity and the leaf value change.
- Our NumPy version matches `scikit-learn`. `scikit-learn` is just faster (pre-sorted features, compiled code) and has more options (`min_samples_leaf`, `ccp_alpha` pruning, ...).
- Why greedy is not optimal (the XOR example) and pruning: see notes 06, sections 5 and 9.